# FIFA ANALYSIS

## CELL 1: Import libraries

In [16]:
import pandas as pd
import os
import sqlite3

## CELL 2: Set file paths 
### Update these paths to match where your CSVs are on your Mac

In [ ]:
# Updating these paths to match where my CSVs are on my device
DATA_RAW = "../data/raw/"
 
paths = {
    "results"      : DATA_RAW + "results.csv",
    "wc_finals"    : DATA_RAW + "List of FIFA World Cup finals.csv",
    "wc_attendance": DATA_RAW + "FIFA World Cup Attendance.csv",
    "wc_awards"    : DATA_RAW + "FIFA World Cup Award.csv",
    "wc_top4"      : DATA_RAW + "Teams reaching the top four.csv",
    "fifa_ranking" : DATA_RAW + "fifa_ranking-2023-07-20.csv",}

for name, path in paths.items():
    status = "Loaded Successfully!" if os.path.exists(path) else "NOT loaded — check path"
    print(f"{name:20s} -> {status}")

## CELL 3: Load all datasets 

In [ ]:
results       = pd.read_csv(paths["results"],       encoding="latin-1")
wc_finals     = pd.read_csv(paths["wc_finals"],     encoding="latin-1")
wc_attendance = pd.read_csv(paths["wc_attendance"], encoding="latin-1")
wc_awards     = pd.read_csv(paths["wc_awards"],     encoding="latin-1")
wc_top4       = pd.read_csv(paths["wc_top4"],       encoding="latin-1")
fifa_ranking  = pd.read_csv(paths["fifa_ranking"],  encoding="latin-1")

print("Datasets loaded successfully!")

## CELL 4: Shape check — how many rows and columns? 

In [ ]:
datasets = {
    "results"       : results,
    "wc_finals"     : wc_finals,
    "wc_attendance" : wc_attendance,
    "wc_awards"     : wc_awards,
    "wc_top4"       : wc_top4,
    "fifa_ranking"  : fifa_ranking,
}
# results.head()
# results.info()
# results.describe()
# results.isnull().sum()
# results['home_team'].value_counts().head(10)
# results['tournament'].unique()

print(f"{'Dataset':<20} {'Rows':>8} {'Columns':>10}")
for name, df in datasets.items():
    print(f"{name:<20} {df.shape[0]:>8,} {df.shape[1]:>10}")

## CELL 5: Column names of every dataset

In [ ]:
for name, df in datasets.items():
    print(f"\n {name}")
    print(df.columns.tolist())

## CELL 6: Null value check — is the data clean? 

In [ ]:
print("MISSING VALUES PER COLUMN\n")
for name, df in datasets.items():
    nulls = df.isnull().sum()
    has_nulls = nulls[nulls > 0]
    if len(has_nulls) == 0:
        print(f"{name}: No nulls")
    else:
        print(f"{name}: Has nulls")
        print(has_nulls)
        print()

## CELL 7: Data type check
### Dates should be datetime, not string — we'll fix this

In [ ]:
# ── CELL 7: Data type check
# Dates should be datetime, not string — we'll fix this
for name, df in datasets.items():
    print(f"\n── {name} dtypes ──")
    print(df.dtypes)

## CELL 8: Convert date columns to datetime

In [ ]:
results["date"] = pd.to_datetime(results["date"])
fifa_ranking["rank_date"] = pd.to_datetime(fifa_ranking["rank_date"])
 
print("Date columns converted")
print(f"results date range     : {results['date'].min().date()}     {results['date'].max().date()}")
print(f"fifa_ranking date range: {fifa_ranking['rank_date'].min().date()}     {fifa_ranking['rank_date'].max().date()}")

## CELL 9: Quick look at each dataset (first 3 rows)

In [ ]:
for name, df in datasets.items():
    print(f"\n{'-'*180}")
    print(f"  {name}")
    print(f"{'-'*180}")
    print(df.head(3).to_string())

## CELL 10: results.csv — tournament breakdown
### This shows you every tournament type in the dataset


In [ ]:
print("All TOURNAMENT types in results.csv:\n")
print(results["tournament"].value_counts().to_string())

## CELL 11: results.csv — filter for key tournaments

In [ ]:
key_tournaments = [
    "FIFA World Cup",
    "Copa América",
    "UEFA Euro",
    "FIFA World Cup qualification",
    "UEFA Euro qualification",
    "CONMEBOL Copa América",         # older name used in some records
]

key_matches = results[results["tournament"].isin(key_tournaments)]
print(f"Key tournament matches: {len(key_matches):,} out of {len(results):,} total")
print()
print(key_matches["tournament"].value_counts())

## CELL 12: fifa_ranking.csv — unique teams and confederations

In [ ]:
print(f"Unique teams in ranking data: {fifa_ranking['country_full'].nunique()}")
print()
print("Teams per confederation:")
print(fifa_ranking.drop_duplicates("country_full")["confederation"].value_counts())

## CELL 13: fifa_ranking — pre-WC snapshots check
### For the ML model we need rankings just before each WC
### Check what dates we have close to each WC June

In [ ]:
# For the ML model we need rankings just before each WC
# Check what dates we have close to each WC June
wc_years = [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]
 
print("Pre-tournament ranking snapshots available:\n")
print(f"{'WC Year':<10} {'Closest ranking date':<25} {'Teams ranked'}")
print("-" * 50)
 
for yr in wc_years:
    may_june = fifa_ranking[(fifa_ranking["rank_date"].dt.year == yr) &
        (fifa_ranking["rank_date"].dt.month.isin([4, 5, 6]))]
    
    if len(may_june) > 0:
        latest = may_june["rank_date"].max()
        n_teams = fifa_ranking[fifa_ranking["rank_date"] == latest].shape[0]
        print(f"{yr:<10} {str(latest.date()):<25} {n_teams}")
    else:
        print(f"{yr:<10} {'No April–June date found':<25} —")

## CELL 14: wc_finals — champions list 

In [ ]:
print("World Cup Champions (1930–2022):\n")
# Drop the messy unnamed index column first
wc_clean = wc_finals.drop(columns=["Unnamed: 0"], errors="ignore")
print(wc_clean[["Year", "Host", "Champion", "Runner_up"]].to_string(index=False))


#### results.csv:
  1. Drop columns: city, id
  2. Ends Nov 2023 — need 2024-2025 update
  3. Copa América encoded as 'Copa AmÃ©rica' — fix encoding
  4. Create: result column (H/A/D) from scores
 
#### wc_finals.csv:
  1. Drop: Unnamed: 0, Score, Score.1 (venue mixed in)
  2. Rename: 'No. _ofteams' -> 'num_teams'
 
#### wc_top4.csv:
  1. 'Germany1' -> 'Germany'
  2. Extract numbers from 'Titles' column (e.g. "5 (1958...)" ->5)
 
#### fifa_ranking.csv:
  1. Drop: previous_points, rank_change
  2. Fix team name mismatches with results.csv (e.g. 'Korea Republic' vs 'South Korea')
  3. Ends July 2023 — need 2024-2026 update for ML
 
# Notebook 02 (SQL cleaning) will fix that.